In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
import pandas as pd
from pandas.core.series import Series
from PIL import Image

In [ ]:
# Constants:

train_path = "../Data/asl_alphabet_train/asl_alphabet_train"
test_path = "../Data/asl_alphabet_test/asl_alphabet_test/"

IMAGE_SIZE = 128
BATCH_SIZE = 64

train_path_df = pd.read_csv("../Data/train_path_datafram.csv")
train_path_df.info()

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(train_path_df['image_path'], train_path_df['label'], test_size = 0.2, random_state=54)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train,  Y_train, test_size = 0.25, random_state=54)


In [ ]:
train_transforms = transforms.Compose([
	transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
	transforms.RandomRotation(15),           # Randomly rotate the image by up to 15 degrees
	transforms.ColorJitter(brightness=0.2),  # Randomly change brightness
	transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Randomly shift the image
	transforms.ToTensor(),
])

# Validation/Test transforms (Clean data)
test_transforms = transforms.Compose([
	transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
	transforms.ToTensor(),
])

In [ ]:
class ASLDataset(Dataset):
	def __init__(self, X: Series, Y: Series, transform=None):
		self.X = X.reset_index(drop = True)
		self.Y = Y.reset_index(drop = True)
		# self.dataframe = dataframe.reset_index(drop=True)
		self.transform = transform
		# Create a mapping from text labels (A, B, del...) to integers (0, 1, 2...)
		self.labels = sorted(self.Y.unique())
		self.idx = {label: idx for idx, label in enumerate(self.labels)}
	
	def __len__(self):
		return len(self.X)
	
	def __getitem__(self, idx):
		# Fetch image path and class string
		img_path = self.X[idx]
		class_name = self.Y[idx]
	       	
		       # Load the image lazily (only when requested)
		image = Image.open(img_path).convert('RGB')
		label_idx = self.idx[class_name]
		
		# Apply the resizing and normalization transforms
		if self.transform:
			image = self.transform(image)
					
		return image, torch.tensor(label_idx, dtype=torch.long)

# 3. Instantiate the Train, Validation, and Test Datasets
train_dataset = ASLDataset(X_train, Y_train, transform=train_transforms)
valid_dataset = ASLDataset(X_valid, Y_valid, transform=test_transforms)
test_dataset = ASLDataset(X_test, Y_test, transform=test_transforms)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 9)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)

In [ ]:
sample_images, sample_labels = next(iter(train_loader))
print(f"Batch Image Shape: {sample_images.shape}")	# [Batch, Channels, Height, Width]
print(f"Batch Label Shape: {sample_labels.shape}")	# Expected: [64]